<a href="https://colab.research.google.com/github/udplabs/okta-ai-poc/blob/feature%2Fa2a/colabs/cross_app_setup_a2a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setting up Okta for A2A Cross App Access

This notebook provides an API-based companion for configuring the [A2A Colab](https://colab.research.google.com/github/udplabs/okta-ai-poc/blob/main/colabs/cross_app_a2a.ipynb).
This notebook is the API-based companion to the [developer guide](https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm) for adding an AI Agent manually. 

## Prerequisites
1. An Okta Preview environment with O4AA enabled. 
2. A general understanding of how to make API calls and use this notebook.

> IMPORTANT
> *This guide utilizes an API token rather than OAuth-based public/private key.*
>
> In a production environment, we strongly encourage the use of `Public key / Private key` authentication method.
> <br>

## Complete Registration Flow (10 Steps)

1. **Create an API Token and Initialize the Notebook**
   1. Create an Okta API token.
   2. Install dependencies, enter your Okta domain and token, and load the notebook utilities.
2. **Create a Client Application**
   1. Create the OIDC client application.
   2. Assign user access.
3. **Create the Client Application's Authorization Server**
   1. Create a custom authorization server.
   2. Create custom scopes.
   3. Create an access policy and rule.
4. **Create the Benefits MCP Resource**
   1. Create its custom authorization server and configure its audience.
   2. Create custom scopes.
   3. Create an access policy and rule.
5. **Register Agent J**
   1. Register the agent.
   2. Optionally assign an owner manually using the linked guide; no API cell is provided.
   3. Configure client registration and its client secret.
   4. Activate Agent J.
   5. Configure the client application as Agent J's caller.
6. **Register Agent K**
   1. Register the agent.
   2. Configure client registration and its client secret.
   3. Activate Agent K.
7. **Create Agent K's Authorization Server and Configure Its Caller**
   1. Create a custom authorization server for Agent K.
   2. Create custom scopes.
   3. Create an access policy and rule.
   4. Configure Agent J as Agent K's caller.
8. **Configure Agent J's Connection to Agent K**
9. **Add the Benefits MCP Resource Connection to Agent K**
10. **Export the Configuration to the A2A Colab**
    1. Print the completed configuration assignments.
    2. Return to Cell 9 of the A2A Colab to initialize and validate the setup.

### Important Training Notes
- This notebook intentionally uses direct HTTP calls instead of an SDK to teach protocol-level behavior.
- This notebook utilizes an API token *for simplicity*. 
  - **In production, we strongly recommend using a public/private key.**
- This notebook creates agents that use a client secret. 
  - **In production, we strongly recommend using a public/private key.**

<br>

---

## STEP 1: Create an API Token

1. Follow the Okta guide for [creating an API token](https://help.okta.com/oie/en-us/content/topics/security/api.htm#security-api__create-okta-api-token) 
2. Paste the API token below. 
3. Enter your Okta domain. 
4. Click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to set the configuration.

In [ ]:
%pip install --upgrade okta-client-python

In [ ]:
# @title { display-mode: "form" }

from pathlib import (
    Path,
)
import importlib
import sys
import requests
from typing import (
    Any,
    TypedDict,
    Literal,
)
from urllib.parse import (
    urlparse,
)
from typing import (
    cast,
)

OKTA_DOMAIN = ""  # @param { type: "string", placeholder: "Enter your entire Okta domain (including https)"}
API_TOKEN = ""  # @param {type: "string", placeholder: "Enter your Okta API token"}

# @markdown _Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!_

# =============== LOAD NOTEBOOK UTILITY FUNCTIONS ==============
# In hosted notebooks, fetch utils.py before importing it. Locally, prefer the checked-out copy.
UTILS_URL = "https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/feature%2Fa2a/utils.py"
repo_root = next(
    (
        p
        for p in (
            Path.cwd(),
            Path.cwd().parent,
        )
        if (p / "utils.py").exists()
    ),
    None,
)

if repo_root is None or repo_root == Path("/content"):
    import requests

    response = requests.get(
        UTILS_URL,
        timeout=15,
    )
    response.raise_for_status()
    Path("utils.py").write_text(response.text)
    repo_root = Path.cwd()

if str(repo_root) not in sys.path:
    sys.path.insert(
        0,
        str(repo_root),
    )

import utils

importlib.reload(utils)

print(f"✅ Utilities loaded from {utils.__file__}")
# ==========================

try:
    utils.validate_config(
        locals(),
        "a2a_setup",
    )
except ValueError as e:
    print(e)
    raise SystemExit("❌ Configuration validation failed. Please fix the above errors and re-run the cell.")


class State(TypedDict):
    api_token: str
    okta_domain: str
    org_id: str
    client_app: utils.types.OidcAppState
    agent_j: utils.types.AIAgentState
    agent_k: utils.types.AIAgentState
    resource: utils.types.AuthorizationServer | None
    output: dict[
        str,
        Any,
    ]


previous_state = globals().get(
    "STATE",
    {},
)

reset_state: bool | None = None

if len(previous_state.keys()) > 0:
    reset_state = previous_state.get("okta_domain") is not None and previous_state.get("okta_domain") != OKTA_DOMAIN

    if reset_state:
        print("🔄 Resetting state due to change in OKTA_DOMAIN.")
        print("❌ You will need to rerun all steps!")
    else:
        print("ℹ️ Retaining previous state.")

DEFAULT_OUTPUT = {
    "OKTA_DOMAIN": OKTA_DOMAIN,
    "DOMAIN_A_ISSUER": "",
    "REDIRECT_URI": "",
    "CLIENT_ID": "",
    "CLIENT_SECRET": "",
    "PRINCIPAL_A_ID": "",
    "PRINCIPAL_A_RESOURCE_URI": "",
    "PRINCIPAL_A_SECRET": "",
    "DOMAIN_B_ISSUER": "",
    "PRINCIPAL_B_ID": "",
    "PRINCIPAL_B_RESOURCE_URI": "",
    "PRINCIPAL_B_SECRET": "",
    "RESOURCE_ISSUER": "",
    "RESOURCE_SERVER_AUDIENCE": "",
}

STATE: State = {
    "api_token": API_TOKEN,
    "okta_domain": OKTA_DOMAIN,
    "org_id": (
        ""
        if reset_state
        else previous_state.get(
            "org_id",
            "",
        )
    ),
    "client_app": (
        {}
        if reset_state
        else previous_state.get(
            "client_app",
            {},
        )
    ),
    "agent_j": (
        {}
        if reset_state
        else previous_state.get(
            "agent_j",
            {},
        )
    ),
    "agent_k": (
        {}
        if reset_state
        else previous_state.get(
            "agent_k",
            {},
        )
    ),
    "resource": (
        None
        if reset_state
        else previous_state.get(
            "resource",
            None,
        )
    ),
    "output": (
        DEFAULT_OUTPUT
        if reset_state
        else previous_state.get(
            "output",
            DEFAULT_OUTPUT,
        )
    ),
}


# ========== HELPER FUNCTIONS ==========
def okta_request(
    path: str,
    method: str = "POST",
    *,
    okta_domain: str | None = OKTA_DOMAIN,
    json: Any = None,
    params: (
        dict[
            str,
            Any,
        ]
        | None
    ) = None,
    timeout: float = 30,
) -> requests.Response:
    """Call the Okta management API using this notebook's API_TOKEN.

    Example: apps = okta_request("GET", "/api/v1/apps", params={"limit": 20}).json()
    For POST or PUT, pass the payload as json=payload.
    HTTP failures raise requests.HTTPError; successful calls return a Response.
    A 204 response has no JSON body. Inspect response.links for pagination;
    this helper returns one page and does not automatically retry mutations.
    API_TOKEN uses SSWS authentication, not an OAuth Bearer access token.
    """
    return utils.okta_request(
        method,
        path,
        okta_domain=okta_domain,
        api_token=API_TOKEN,
        json=(
            {key: value for key, value in json.items() if value is not None}
            if isinstance(
                json,
                dict,
            )
            else json
        ),
        params=params,
        timeout=timeout,
    )


def get_okta_partition(
    okta_domain: str | None = None,
) -> str:
    if okta_domain is None:
        okta_domain = STATE.get(
            "okta_domain",
            OKTA_DOMAIN,
        )

    """Infer ORN partition from Okta domain."""
    hostname = urlparse(okta_domain).hostname or ""
    if hostname.endswith(".oktapreview.com"):
        return "oktapreview"
    if hostname.endswith(".okta-emea.com"):
        return "okta-emea"
    if hostname.endswith(".okta.com"):
        return "okta"
    raise ValueError(f"Unsupported Okta domain for ORN partition derivation: {okta_domain}")


def get_org_id():
    if not STATE.get("org_id"):
        response = okta_request(
            path=f"/.well-known/okta-organization",
            method="GET",
        )
        org_id = response.json().get("id")
        STATE["org_id"] = org_id
    return STATE["org_id"]


def get_orn(
    service: Literal[
        "directory",
        "idp",
        "workflow",
        "governance",
    ],
    object_type: Literal[
        "apps",
        "workload-principals",
        "groups",
        "users",
        "resource-servers",
        "authorization_servers",
        "identity_provider",
    ],
    *,
    partition: str | None = None,
    org_id: str | None = None,
    app_name: str | None = None,
    object_id: str | None = None,
) -> str:
    orn = f"orn:{partition or get_okta_partition()}:{service}:{org_id or get_org_id()}:{object_type}"

    if app_name:
        orn += f":{app_name}"

    if object_id:
        orn += f":{object_id}"
    return orn

## Step 2: Create a Client Application

In this section you will create a custom OIDC application and then assign a user to the application. 

*Learn more about creating a custom OIDC application via the UI [here](https://help.okta.com/oie/en-us/content/topics/apps/apps_app_integration_wizard_oidc.htm).*

### 1. Create a Client Application

The following code will create a custom OIDC application in Okta. Click `Show Code` to see the details of the API request or to modify the parameters (at your own risk).

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

CLIENT_APP_NAME = "Chat App"  # @param { type: "string", placeholder: "Enter a name for your client/web application." }
# @markdown <sup><code>label</code></sup>
# @markdown <br>
# @markdown <sub><em><b>NOTE:</b> This notebook currently only supports `client_secret_basic` authentication at this time.</em></sub>


def create_app(name: str) -> utils.types.OidcApp:
    print(f"⏳ Creating client application '{name}'...")

    result = okta_request(
        path="/api/v1/apps",
        json={
            "label": name,
            "signOnMode": "OPENID_CONNECT",
            "credentials": {"oauthClient": {"token_endpoint_auth_method": "client_secret_basic"}},
            "name": "oidc_client",
            "settings": {
                "oauthClient": {
                    "application_type": "web",
                    "grant_types": ["authorization_code", "refresh_token"],
                    "redirect_uris": ["http://localhost:8080/authorization-code/callback"],
                    "response_types": ["code"],
                }
            },
        },
    )

    data = cast(utils.types.OidcApp, result.json())

    print(f"✅ Client application created: {data.get("label")}")

    return data


client_app = STATE.get("client_app", {}).get("app", {})
client_id = client_app.get("id", None)
client_secret = client_app.get("credentials", {}).get("oauthClient", {}).get("client_secret", None)

if not client_app:
    client_app = create_app(CLIENT_APP_NAME)

    client_id = client_app.get("id", None)
    client_secret = client_app.get("credentials", {}).get("oauthClient", {}).get("client_secret", None)

    STATE["output"]["CLIENT_ID"] = client_id
    STATE["output"]["CLIENT_SECRET"] = client_secret
    STATE["output"]["REDIRECT_URI"] = client_app.get("settings").get("oauthClient").get("redirect_uris")[0]

elif client_app.get("id") and client_app.get("label") == CLIENT_APP_NAME:
    print(f"⚠️ Client application '{CLIENT_APP_NAME}' already exists!")
    print("Change the CLIENT_APP_NAME value if you would like to create a new application.")

if client_app.get("orn") is None:
    print(f"⚠️ Client application '{CLIENT_APP_NAME}' does not have an ORN!")
    print("⏳ Generating ORN...")

    client_app["orn"] = get_orn(
        service="idp",
        object_type="apps",
        app_name="oidc_client",
        object_id=client_app.get("id"),
    )

STATE["client_app"]["app"] = cast(utils.types.OidcApp, client_app)

print("   ‼️ Copy the following values to your A2A Colab Notebook!")
print(f"    → CLIENT_ID: {client_id}")
print(f"    → CLIENT_SECRET: {client_secret}")

### 2. Assign user access

Before a user can authenticate against an application, they must be assigned access.

In [ ]:
# @title { display-mode: "form" }

from urllib.parse import quote

USERNAME = ""  # @param { type: "string", placeholder: "Enter a username you are able to authenticate with in your tenant." }


def assign_app_access(username: str) -> None:
    username = username.strip()
    if not username:
        raise ValueError("Enter USERNAME before running this cell.")

    app_id = STATE.get("client_app", {}).get("app", {}).get("id")
    if not app_id:
        raise ValueError("Create the client application in the previous step first.")

    user = okta_request(
        path=f"/api/v1/users/{quote(username, safe='')}",
        method="GET",
    ).json()
    user_id = user.get("id")
    if not user_id:
        raise ValueError("The user lookup did not return an Okta user ID.")

    assignment_path = f"/api/v1/apps/{app_id}/users/{user_id}"
    try:
        okta_request(path=assignment_path, method="GET")
    except utils.OktaApiError as error:
        if error.response.status_code != 404:
            raise
    else:
        print(f"User '{username}' already has access to the client application.")
        return

    okta_request(
        path=f"/api/v1/apps/{app_id}/users",
        method="POST",
        json={"id": user_id, "scope": "USER"},
    )
    print(f"✅ User '{username}' assigned to the client application successfully.")


assign_app_access(USERNAME)

## Step 3: Create a Custom Authorization Server

TODO


### 1. Create a custom authorization Server

TODO

https://help.okta.com/oie/en-us/content/topics/security/api-config-auth-server.

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

CLIENT_AUTH_SERVER_NAME = "Chat App"  # @param { type: "string", placeholder: "Enter a name for your client/web application\'s authorization server." }
# @markdown <sup><code>name</code></sup>


def create_auth_server(
    name: str,
    description: str,
    audiences: list[str],
) -> utils.types.AuthorizationServer:
    client_authz_result = okta_request(
        path="/api/v1/authorizationServers",
        json={
            "name": name,
            "description": description,
            "audiences": audiences,
        },
    )

    auth_server_response = client_authz_result.json()

    if not isinstance(
        auth_server_response,
        dict,
    ):
        raise ValueError("Authorization Server response was not a JSON object.")

    resource_server = cast(
        utils.types.AuthorizationServer,
        auth_server_response,
    )

    print(f"✅ Custom Authorization Server Created: {resource_server.get('name')}")

    # Generate ORN. We will need it later.
    if not resource_server.get("orn"):
        resource_server.update(
            {
                "orn": get_orn(
                    service="idp",
                    object_type="authorization_servers",
                    object_id=resource_server.get("id"),
                )
            }
        )

    return resource_server


try:
    client_app_resource_server = create_auth_server(
        name=CLIENT_AUTH_SERVER_NAME,
        description=f"{CLIENT_AUTH_SERVER_NAME} Authorization Server",
        audiences=["api://default"],
    )
except utils.OktaApiError as error:
    duplicate_name = error.response.status_code == 400 and error.error_code == "E0000001" and any(cause.get("errorSummary", "").startswith("name:") and "already exists" in cause.get("errorSummary", "") for cause in error.error_causes)
    if not duplicate_name:
        raise
    print(f"Authorization server '{CLIENT_AUTH_SERVER_NAME}' already exists. Skipping creation; existing state values were retained.")
    client_app_resource_server = STATE["client_app"].get("resource_server")
    if not client_app_resource_server:
        print("No authorization server is saved in notebook state. Load the existing server before continuing with scopes and policies.")
else:
    STATE["client_app"]["resource_server"] = client_app_resource_server
    STATE["output"]["DOMAIN_A_ISSUER"] = client_app_resource_server.get("issuer")


print("   ‼️ Copy the following values to your A2A Colab Notebook!")
print(f"    → DOMAIN_A_ISSUER: {STATE['output'].get('DOMAIN_A_ISSUER', '')}")
if client_app_resource_server:
    print(f"    → OR use the Auth Server ID: {client_app_resource_server.get('id')}")

### 2. Create a custom scope

TODO

https://help.okta.com/oie/en-us/content/topics/security/api-config-scopes.htm

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

# @markdown <sub><em>Separate multiple scope names with commas.</em></sub>
CLIENT_AUTH_SERVER_SCOPES = "agent:invoke"  # @param { type: "string", placeholder: "Enter scope names for your Client\'s required scopes." }
# @markdown <sup><code>name</code> <code>displayName</code></sup>
# @markdown <br><sub><em>Separate multiple scope descriptions with commas in the same order as the name(s)</em></sub>
CLIENT_AUTH_SERVER_SCOPE_DESCRIPTIONS = "Invoke AI Agent"  # @param { type: "string", placeholder: "Enter a description for your Client\'s required scope." }
# @markdown <sup><code>description</code></sup>


def create_scopes(
    auth_server: utils.types.AuthorizationServer,
    scopes: str,
    descriptions: str,
    displayNames: str | None = None,
) -> str:

    scope_list = [s.strip() for s in scopes.split(",")]
    description_list = [d.strip() for d in descriptions.split(",")]
    displayName_list = [dn.strip() for dn in displayNames.split(",")] if displayNames else scope_list

    try:
        scopes_output = []

        for (
            scope_index,
            name,
        ) in enumerate(scope_list):
            description = description_list[scope_index] or name if scope_index < len(description_list) else name
            displayName = displayName_list[scope_index] or name if scope_index < len(displayName_list) else name
            result = okta_request(
                path=f"/api/v1/authorizationServers/{auth_server.get('id')}/scopes",
                json={
                    "name": name,
                    "description": description,
                    "consent": "REQUIRED",
                    "displayName": displayName or name,
                },
            )
            scope_result = cast(
                dict[
                    str,
                    Any,
                ],
                result.json(),
            )
            scope_name = cast(
                str,
                scope_result.get("name"),
            )
            print(f"✅ Custom Scope Created: {scope_name}")

            scopes_output.append(scope_name)

        return ", ".join(scopes_output)

    except Exception as e:
        print(f"❌ Error creating custom scopes: {e}")
        raise


if (client_app_resource_server := STATE["client_app"].get("resource_server")) is None:
    raise ValueError("❌ Client application resource server must be created before setting up its scopes. " "Rerun the client authorization server setup cell.")

client_app_scopes = create_scopes(
    auth_server=client_app_resource_server,
    scopes=CLIENT_AUTH_SERVER_SCOPES,
    descriptions=CLIENT_AUTH_SERVER_SCOPE_DESCRIPTIONS,
    displayNames=CLIENT_AUTH_SERVER_SCOPES,
)

STATE["output"]["CLIENT_SCOPES"] = f"{client_app_scopes} openid profile email offline_access"

print(f"   ‼️ Copy the following values to your A2A Colab Notebook!")
print(f"     → CLIENT_SCOPES: {STATE['output'].get('CLIENT_SCOPES')}")

### 3. Create an access policy & rule

TODO

https://help.okta.com/oie/en-us/content/topics/security/api-config-access-policies.htm

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

CLIENT_AUTH_SERVER_POLICY_NAME = "Client App Policy"  # @param { type: "string", placeholder: "Enter a name for your Client\'s access policy." }
# @markdown <sup><code>name</code> <code>displayName</code></sup>
CLIENT_AUTH_SERVER_POLICY_DESCRIPTION = "Policy for Client App authorization server"  # @param { type: "string", placeholder: "Enter a description for your Client\'s access policy rule." }
CLIENT_AUTH_SERVER_RULE_NAME = "Invoke Agent"  # @param { type: "string", placeholder: "Enter a name for your Client\'s access policy rule." }
# @markdown <sup><code>description</code></sup>


# ========== HELPER FUNCTION ==========
def create_policy(
    auth_server: utils.types.AuthorizationServer,
    name: str,
    description: str,
    rule_name: str,
    scopes: list[str],
) -> str:
    auth_server_id = auth_server.get("id")

    if not auth_server_id:
        raise ValueError("Authorization server is missing its ID.")

    access_policy_id = auth_server.get("resource_access_policy_id")

    if not access_policy_id:
        policy_result = okta_request(
            path=f"/api/v1/authorizationServers/{auth_server_id}/policies",
            json={
                "type": "OAUTH_AUTHORIZATION_POLICY",
                "name": name,
                "description": description,
                "conditions": {"clients": {"include": ["ALL_CLIENTS"]}},
                "status": "ACTIVE",
            },
        )
        policy_body = policy_result.json()
        access_policy_id = policy_body.get("id")
        if (
            not isinstance(
                access_policy_id,
                str,
            )
            or not access_policy_id
        ):
            raise ValueError("Created access policy response did not include an ID.")

        auth_server["resource_access_policy_id"] = access_policy_id
        print(f"✅ Access Policy Created: {policy_body.get('name')}")
    else:
        print(f"ℹ️ Access policy already created: {access_policy_id}")

    rule_result = okta_request(
        path=f"/api/v1/authorizationServers/{auth_server_id}/policies/{access_policy_id}/rules",
        json={
            "type": "RESOURCE_ACCESS",
            "name": rule_name,
            "conditions": {
                "people": {
                    "users": {
                        "include": [],
                        "exclude": [],
                    },
                    "groups": {
                        "include": ["EVERYONE"],
                        "exclude": [],
                    },
                },
                "grantTypes": {
                    "include": [
                        "urn:ietf:params:oauth:grant-type:token-exchange",
                        "urn:ietf:params:oauth:grant-type:jwt-bearer",
                        "authorization_code",
                    ],
                },
                "scopes": {"include": scopes},
            },
            "status": "ACTIVE",
        },
    )

    print(f"✅ Access Policy Rule Created: {rule_result.json().get('name')}")
    return access_policy_id


# ========== END HELPER FUNCTION ==========

if (client_app_resource_server := STATE["client_app"].get("resource_server")) is None:
    raise ValueError("Client app resource server not found. Create it first.")

client_app_resource_server_policy_id = create_policy(
    auth_server=client_app_resource_server,
    name=CLIENT_AUTH_SERVER_POLICY_NAME,
    description=CLIENT_AUTH_SERVER_POLICY_DESCRIPTION,
    rule_name=CLIENT_AUTH_SERVER_RULE_NAME,
    scopes=STATE["output"].get("CLIENT_SCOPES", "[*]").strip().strip("[]").replace(",", " ").split(),
)

client_app_resource_server["resource_access_policy_id"] = client_app_resource_server_policy_id

STATE["client_app"]["resource_server"] = client_app_resource_server

## Step 4: Create an MCP Resource

TODO

### 1. Create a custom authorization Server

TODO

https://help.okta.com/oie/en-us/content/topics/security/api-config-auth-server.

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

MCP_RESOURCE_SERVER_NAME = "Benefits MCP Server2"  # @param { type: "string", placeholder: "Enter a name for your MCP resource\'s authorization server." }
# @markdown <sup><code>name</code></sup>
MCP_RESOURCE_SERVER_AUDIENCE = "https://benefits.streamward.com"  # @param { type: "string", placeholder: "Enter an audience value for your MCP resource." }
# @markdown <sup><code>audiences</code></sup>

mcp_resource_server = create_auth_server(
    name=MCP_RESOURCE_SERVER_NAME,
    description=f"{MCP_RESOURCE_SERVER_NAME} Authorization Server",
    audiences=[MCP_RESOURCE_SERVER_AUDIENCE],
)

STATE["resource"] = mcp_resource_server
STATE["output"]["RESOURCE_ISSUER"] = mcp_resource_server.get("issuer")
STATE["output"]["RESOURCE_SERVER_AUDIENCE"] = mcp_resource_server.get(
    "audiences",
    [MCP_RESOURCE_SERVER_AUDIENCE],
)[0]

print("   ‼️ Copy the following values to your A2A Colab Notebook!")
print(f"    → RESOURCE_ISSUER: {mcp_resource_server.get('issuer')}")
print(f"        (or use the resource server ID): {mcp_resource_server.get('id')}")
print(f"    → RESOURCE_SERVER_AUDIENCE: {mcp_resource_server.get('audiences', [MCP_RESOURCE_SERVER_AUDIENCE])[0]}")

### 2. Create a custom scope

TODO

https://help.okta.com/oie/en-us/content/topics/security/api-config-scopes.htm

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

# @markdown <sub><em>Separate multiple scope names with commas.</em></sub>
MCP_RESOURCE_SERVER_SCOPES = "mcp:read"  # @param { type: "string", placeholder: "Enter scope names for your MCP resource\'s required scopes." }
# @markdown <sup><code>name</code> <code>displayName</code></sup>
# @markdown <br><sub><em>Separate multiple scope descriptions with commas in the same order as the name(s)</em></sub>
MCP_RESOURCE_SERVER_SCOPE_DESCRIPTIONS = "Read-only access"  # @param { type: "string", placeholder: "Enter descriptions for your MCP resource\'s required scopes." }
# @markdown <sup><code>description</code></sup>

if (mcp_resource_server := STATE.get("resource")) is None:
    raise ValueError("❌ MCP resource server must be created before setting up its scopes. " "Rerun the MCP authorization server setup cell.")

mcp_resource_server_scopes = create_scopes(
    auth_server=mcp_resource_server,
    scopes=MCP_RESOURCE_SERVER_SCOPES,
    descriptions=MCP_RESOURCE_SERVER_SCOPE_DESCRIPTIONS,
    displayNames=MCP_RESOURCE_SERVER_SCOPES,
)

STATE["output"]["PRINCIPAL_B_SCOPES"] = mcp_resource_server_scopes

print(f"   ‼️ Copy the following values to your A2A Colab Notebook!")
print(f"     → PRINCIPAL_B_SCOPES: {STATE['output'].get('PRINCIPAL_B_SCOPES')}")

### 3. Create an access policy & rule

TODO

https://help.okta.com/oie/en-us/content/topics/security/api-config-access-policies.htm

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

MCP_RESOURCE_SERVER_POLICY_NAME = "Benefits MCP Policy"  # @param { type: "string", placeholder: "Enter a name for your MCP resource\'s access policy." }
# @markdown <sup><code>name</code> <code>displayName</code></sup>
MCP_RESOURCE_SERVER_POLICY_DESCRIPTION = "Policy for Benefits MCP server"  # @param { type: "string", placeholder: "Enter a description for your MCP resource\'s access policy rule." }
MCP_RESOURCE_SERVER_RULE_NAME = "Read-only access"  # @param { type: "string", placeholder: "Enter a name for your MCP resource\'s access policy rule." }
# @markdown <sup><code>description</code></sup>

if (mcp_resource_server := STATE["resource"]) is None:
    raise ValueError("MCP resource server not found in STATE. Make sure you have created the MCP resource server before setting up the policy.")

mcp_resource_server_policy_id = create_policy(
    auth_server=mcp_resource_server,
    name=MCP_RESOURCE_SERVER_POLICY_NAME,
    description=MCP_RESOURCE_SERVER_POLICY_DESCRIPTION,
    rule_name=MCP_RESOURCE_SERVER_RULE_NAME,
    scopes=[
        scope.strip()
        for scope in STATE["output"]
        .get(
            "PRINCIPAL_B_SCOPES",
            "[*]",
        )
        .strip()
        .strip("[]")
        .split(",")
        if scope.strip()
    ],
)

mcp_resource_server["resource_access_policy_id"] = mcp_resource_server_policy_id

STATE["resource"] = mcp_resource_server

## Step 5: Register Agent J

TODO

### 1. Create the agent

https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>
# @markdown You can change these values but we do not recommend it as it might make the setup inconsistent with the rest of the notebook.

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

import time

PRINCIPAL_A_NAME = "Agent J"  # @param { type: "string", placeholder: "Enter a name for Principal A." }
# @markdown <sup><code>profile.name</code></sup>
PRINCIPAL_A_DESCRIPTION = "Special Agent J"  # @param { type: "string", placeholder: "Enter a description for Principal A." }
# @markdown <sup><code>profile.description</code></sup>
PRINCIPAL_A_RESOURCE_URI = "https://agent_j"  # @param { type: "string", placeholder: "Enter the resource URI for Principal A." }


def create_agent(
    agentProfile: utils.types.AIAgentProfile,
    resource_uri: str | None = None,
) -> utils.types.AIAgentOperation:
    print(f"⏳ Creating agent with profile: {agentProfile}{ f"and resource URI: {resource_uri}" if resource_uri else ''}")

    create_result = okta_request(
        path="/workload-principals/api/v1/ai-agents",
        json={
            "profile": agentProfile,
            "resourceUrl": resource_uri,
        },
    )

    operation_url = create_result.headers.get("Location")

    if not isinstance(
        operation_url,
        str,
    ):
        raise ValueError("Failed to create a new agent. Location header is missing.")

    print("⌛️ AI Agent registration started. Waiting for completion...")

    return cast(
        utils.types.AIAgentOperation,
        {
            "status": "IN_PROGRESS",
            "url": operation_url,
        },
    )


def register_agent_v2(
    agent: utils.types.AIAgent,
    *,
    name: str | None = None,
    description: str | None = None,
    resource_uri: str | None = None,
) -> utils.types.AIAgent:

    current_operation = agent.get("current_operation")
    current_operation_url = current_operation.get("url") if current_operation else None
    agent_id = agent.get("id")
    agent_profile = agent.get("profile")
    agent_name = agent_profile.get("name") if agent_profile else None
    agent_description = agent_profile.get("description") if agent_profile else None

    # Check if there is an existing agent ID
    if agent_id is None:
        # Create a new agent
        print("⚠️ No existing agent found.")

        current_operation = create_agent(
            agentProfile={
                "name": name,
                "description": description,
            },
            resource_uri=resource_uri,
        )

        agent.update({"current_operation": current_operation})

        current_operation_url = current_operation.get("url")
    else:
        print(f"⚠️ Agent with ID {agent_id} already exists.")

    # Check if there is an operation_url
    if current_operation_url is None:
        print("Agent has no ongoing operation.")

        agent_links = agent.get(
            "_links",
            {},
        )
        agent_resource_link = agent_links.get(
            "resourceServer",
            {},
        )
        resource_link_href = (
            agent_resource_link.get("href")
            if isinstance(
                agent_resource_link,
                dict,
            )
            else None
        )
        agent_resource_uri = resource_link_href.rstrip("/").split("/")[-1] if resource_link_href else None

        # Check if the agent already has a resource server.
        # If it does but the incoming resource_uri is not the same, we need to create a new agent.
        if agent_resource_uri and resource_uri and agent_resource_uri != resource_uri:
            print(f"❌ Agent resource URI '{agent_resource_uri}' does not match the provided resource URI '{resource_uri}'.")
            print("⚠️ A new agent must be created!")

            current_operation = create_agent(
                agentProfile={
                    "name": name,
                    "description": description,
                },
                resource_uri=resource_uri,
            )

            agent.update({"current_operation": current_operation})

            current_operation_url = current_operation.get("url")
        elif (name and agent_name != name) or (description and agent_description != description):
            print("Agent profile does not match the provided input values. Updating agent profile.")
            # Update agent (including the resource_uri)
        else:
            print("Agent profile matches the provided input values. No update needed.")
            # No action needed as the agent profile matches the provided input values.

    print(f"Agent has an ongoing operation at {current_operation_url}.")
    # Poll the operation
    current_operation = cast(
        utils.types.AIAgentOperation,
        complete_operation(current_operation_url),
    )

    print(f"✅ AI Agent operation completed!")

    agent_self_href = current_operation.get("resource").get("_links").get("self").get("href")

    if not agent_self_href:
        raise ValueError("Completed operation did not include an agent self link.")

    print(f"⌛️ Fetching agent details...")

    response = okta_request(
        path=agent_self_href,
        method="GET",
        okta_domain=None,
    )

    agent.update(
        cast(
            utils.types.AIAgent,
            response.json(),
        )
    )

    # At this point we should remove any current operation as we should have an agent.
    # This prevents an infinite loop!
    agent.pop(
        "current_operation",
        None,
    )
    # If an agent exists (either provided or from create/update) check for a URN
    if not agent.get("orn"):
        print("⏳ Generating missing ORN for the agent...")
        agent.update(
            {
                "orn": get_orn(
                    service="directory",
                    object_type="workload-principals",
                    app_name="ai-agents",
                    object_id=agent_id,
                )
            }
        )

    # At this point we should have an agent!
    return agent


# ========== HELPER FUNCTIONS ==========
def register_agent(
    agent: utils.types.AIAgent,
    *,
    name: str | None = None,
    description: str | None = None,
    resource_uri: str | None = None,
) -> utils.types.AIAgent:
    """Reuse, resume, or register in place; None overrides retain saved values."""

    current_name = profile.get("name") if (profile := agent.get("profile")) else None
    current_description = profile.get("description") if profile else current_name
    current_resource_uri = agent.get("resource_uri")

    if has_changed := (agent.get("id") and ((name and name != current_name) or (description and description != current_description) or (resource_uri and resource_uri != current_resource_uri))):
        print("⚠️ Notebook input values changed! Clearing existing agent data & registering new agent.")

    operation_url = (
        None
        if has_changed
        else (
            current_operation.get("url")
            if (
                (current_operation := agent.get("current_operation"))
                and current_operation.get("status")
                in [
                    "IN_PROGRESS",
                    "SCHEDULED",
                ]
            )
            else None
        )
    )

    agent_missing = False
    output = STATE.get("output", {})
    has_saved_secret = any(output.get(f"{principal}_ID") == agent.get("id") and output.get(f"{principal}_SECRET") for principal in ("PRINCIPAL_A", "PRINCIPAL_B"))
    if agent.get("id") and not has_changed and not operation_url and not has_saved_secret:
        try:
            okta_request(
                path=f"/workload-principals/api/v1/ai-agents/{agent.get('id')}",
                method="GET",
            )
        except utils.OktaApiError as error:
            if error.response.status_code != 404:
                raise
            agent_missing = True
            print("The cached agent was not found in Okta. Registering a replacement.")

    if has_changed or agent_missing or not agent.get("id"):

        agent = {}

        new_name = name or current_name
        new_description = description or current_description or new_name

        result = okta_request(
            path="/workload-principals/api/v1/ai-agents",
            json={
                "profile": {
                    "name": new_name,
                    "description": new_description,
                },
                "resourceUrl": resource_uri,
            },
        )

        if (
            not isinstance(
                operation_url := result.headers.get("Location"),
                str,
            )
            or not operation_url.strip()
        ):
            raise ValueError("Create AI Agent response did not include a Location header.")

        print("⏳ AI Agent registration started. Waiting for completion...")

        agent["current_operation"] = cast(
            utils.types.AIAgentOperation,
            {
                "status": "IN_PROGRESS",
                "url": operation_url,
            },
        )

    if operation_url:
        operation = cast(
            utils.types.AIAgentOperation,
            complete_operation(operation_url),
        )

        print(f"✅ AI Agent registration completed!")

        agent_url = resource_self.get("href") if (resource_self := (resource_links.get("self") if (resource_links := operation.get("resource").get("_links")) else None)) else None

        if not agent_url:
            raise ValueError("Completed operation did not include an agent self link.")

        print(f"⌛️ Fetching agent details...")

        response = okta_request(
            path=agent_url,
            method="GET",
            okta_domain=None,
        )

        agent.update(
            cast(
                utils.types.AIAgent,
                response.json(),
            )
        )
        agent["resource_uri"] = resource_uri if resource_uri else (current_resource_uri if current_resource_uri else "")

    else:
        print("Reusing the saved agent registration.")

    agent.pop(
        "current_operation",
        None,
    )

    # Generate ORN if not present. We will need it later
    if not agent.get("orn"):
        print("⏳ Generating missing ORN for the agent...")
        agent.update(
            {
                "orn": get_orn(
                    service="directory",
                    object_type="workload-principals",
                    app_name="ai-agents",
                    object_id=agent.get("id"),
                )
            }
        )

    return agent


def complete_operation(
    operation_url: str | None,
    max_poll_attempts: int = 60,
    poll_inverval_seconds: int = 2,
) -> utils.types.Operation:
    if not operation_url:
        raise ValueError("An operation_url must be provided.")

    try:
        print(f"Polling operation URL: {operation_url} (max attempts: {max_poll_attempts}, interval: {poll_inverval_seconds}s)")

        for attempt in range(
            1,
            max_poll_attempts + 1,
        ):
            operation_response = okta_request(
                path=operation_url,
                method="GET",
                okta_domain=None,
            )

            operation = cast(
                utils.types.Operation,
                operation_response.json(),
            )

            status = operation.get("status")

            print(f"Operation status: {status} ({attempt}/{max_poll_attempts})")

            if status == "COMPLETED":
                return operation

            if status in {
                "FAILED",
                "CANCELED",
            }:
                raise RuntimeError(f"Operation ended with status {status}: {operation}")

            if attempt < max_poll_attempts:
                time.sleep(poll_inverval_seconds)

        raise TimeoutError(f"Operation did not complete after " f"{max_poll_attempts * poll_inverval_seconds} seconds. " "Rerun this cell to continue polling.")
    except Exception as e:
        print(f"❌ Error completing operation:\n   {e}")
        raise


# ========== END HELPER FUNCTIONS ==========

agent_j = STATE.get("agent_j")
if agent_j is None:
    agent_j = cast(
        utils.types.AIAgentState,
        {},
    )
    STATE["agent_j"] = agent_j

try:
    agent_j = cast(
        utils.types.AIAgentState,
        register_agent(
            agent_j,
            name=PRINCIPAL_A_NAME,
            description=PRINCIPAL_A_DESCRIPTION,
            resource_uri=PRINCIPAL_A_RESOURCE_URI,
        ),
    )

    STATE["agent_j"] = agent_j
finally:
    if agent_j.get("id") != STATE["output"].get("PRINCIPAL_A_ID"):
        STATE["output"]["PRINCIPAL_A_SECRET"] = ""

    STATE["output"]["PRINCIPAL_A_ID"] = agent_j.get(
        "id",
        "",
    )
    STATE["output"]["PRINCIPAL_A_RESOURCE_URI"] = agent_j.get(
        "resource_uri",
        "",
    )

print("   ‼️ Copy the following values to your A2A Colab Notebook!")
print(f"    → PRINCIPAL_A_ID: {agent_j.get('id')}")
print(f"    → PRINCIPAL_A_RESOURCE_URI: {agent_j.get('resource_uri')}")

### 2. Assign an owner (optional)

https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm#:~:text=Assign%20owners%20(optional)

### 3. Configure client registration

TODO

https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm#:~:text=Add%20client%20registration%20details

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>
# @markdown <br><br>
# @markdown You can change these values but we do not recommend it as it might make the setup inconsistent with the rest of the notebook.
# @markdown <br><br>
# @markdown <em>Only <code>client_secret_basic</code> is supported at this time.</em>
# @markdown <br><br>
# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

PRINCIPAL_A_AUTH_METHOD = "client_secret_basic"  # @param ["client_secret_basic"]
# @markdown <sup><code>client.token_endpoint_auth_method</code></sup>


def create_credentials(
    id: str,
) -> str:
    if not id or not id.strip():
        raise ValueError("An agent ID is required. Run agent registration first.")

    output = STATE.get("output", {})
    for principal in ("PRINCIPAL_A", "PRINCIPAL_B"):
        saved_secret = output.get(f"{principal}_SECRET")
        if output.get(f"{principal}_ID") == id and saved_secret:
            print(f"Reusing the client secret saved in notebook state for principal {id}.")
            return saved_secret

    secrets_path = f"/workload-principals/api/v1/ai-agents/{id}/credentials/secrets"
    credentials = okta_request(path=secrets_path, method="GET").json()
    if credentials:
        credential_result = next(
            (credential for credential in reversed(credentials) if credential.get("client_secret")),
            None,
        )
        if credential_result:
            print(f"Retrieved an existing client secret for principal {id}.")
            return credential_result["client_secret"]
        raise ValueError(f"No readable client secret was returned for principal {id}. " "Keep any previously copied secret; existing credentials were not rotated. " "Check the agent's credentials in Okta before creating a replacement.")

    print(f"Creating credentials for principal {id}...")
    created_credential = okta_request(
        path=secrets_path,
        method="POST",
        json={},
    )
    credential_result = created_credential.json() if created_credential.content else {}
    client_secret = credential_result.get("client_secret")
    if not client_secret:
        credentials = okta_request(path=secrets_path, method="GET").json()
        client_secret = next(
            (credential["client_secret"] for credential in reversed(credentials) if credential.get("client_secret")),
            None,
        )
    if not client_secret:
        raise ValueError(f"The new credential for principal {id} did not return a readable client secret.")

    client_data = okta_request(
        path=f"/oauth2/v1/clients/{id}",
        method="GET",
    ).json()
    client_data.update(
        {
            "client_secret": client_secret,
            "client_secret_expires_at": 0,
            "token_endpoint_auth_method": "client_secret_basic",
        }
    )
    okta_request(
        path=f"/oauth2/v1/clients/{id}",
        method="PUT",
        json=client_data,
    )
    print(f"Client secret configured for principal {id}.")
    return client_secret


assert isinstance(
    principal_a_id := STATE["output"].get("PRINCIPAL_A_ID"),
    str,
), "PRINCIPAL_A_ID is missing!"

principal_a_secret = create_credentials(principal_a_id)
STATE["output"]["PRINCIPAL_A_SECRET"] = principal_a_secret

print(f"   ‼️ Copy the following values to your A2A Colab Notebook!")
print(f"     → PRINCIPAL_A_SECRET: {principal_a_secret}")

### 4. Activate Agent J

TODO

https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm#:~:text=Activate%20an%20AI%20agent



In [ ]:
# ========= HELPER FUNCTION ==========
def activate_agent(
    id: str,
):

    print(f"⏳ Activating agent: {id}")

    response = okta_request(
        path=f"/workload-principals/api/v1/ai-agents/{id}/lifecycle/activate",
        method="POST",
    )

    if response.status_code == 202:
        print("✅ Agent activated successfully.")
    else:
        print(f"❌Failed to activate agent. Status code: {response.status_code}")


# ========== END HELPER FUNCTION ==========

assert isinstance(
    principal_a_id := STATE["output"].get("PRINCIPAL_A_ID"),
    str,
), "PRINCIPAL_A_ID is missing!"

activate_agent(principal_a_id)

### 5. Configure Agent Caller

TODO

https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm#:~:text=Configure%20machine%20access

In [ ]:
# @title { display-mode: "form" }
# @markdown Helper functions for creating A2A resources, delegation links, and connections. Click 'Show Code' to see the implementation.


# ========== HELPER FUNCTIONS ==========
def create_a2a_resource(
    wlp_id: str,
    auth_server_orn: str,
):

    print(f"⌛️ Creating A2A resource with ORN: {auth_server_orn}")

    response = okta_request(
        path=f"/resource-servers/api/v1/a2a-servers/{wlp_id}/authorization-servers",
        method="POST",
        json={
            "orn": auth_server_orn,
            "type": "OKTA",
        },
    )

    if response.status_code not in [
        202,
        204,
    ]:
        raise ValueError(f"❌ Failed to create A2A resource. Status code: {response.status_code}, Response: {response.text}")

    print(f"⌛️ Waiting for operation to complete...")

    complete_operation(response.headers.get("Location"))

    print(f"✅ Resource created successfully: {auth_server_orn}")


def create_delegation_link(
    client_orn: str,
    resource_orn: str,
    *,
    auth_server_orn: str | None = None,
    token_type: Literal[
        "ID_TOKEN",
        "ACCESS_TOKEN",
    ] = "ACCESS_TOKEN",
):
    print(f"⌛️ Creating delegation link from client {client_orn} to resource {resource_orn}{f' with auth server {auth_server_orn}' if auth_server_orn else ''}")

    okta_request(
        path="/workload-principals/api/v1/delegation-links",
        method="POST",
        json={
            "from": {
                "type": "OKTA_AUTHORIZATION_SERVER",
                "clientOrn": client_orn,
                "tokenType": token_type,
            },
            "to": {
                "resourceOrn": resource_orn,
                "authorizationServerOrn": auth_server_orn,
            },
        },
    )

    print(f"✅ Delegation link created successfully!")


def create_connection(
    wlp_id: str,
    connection_type: Literal[
        "IDENTITY_ASSERTION_A2A_SERVER",
        "IDENTITY_ASSERTION_CUSTOM_AS",
        "IDENTITY_ASSERTION_APP_INSTANCE",
    ],
    scopes: list[str],
    *,
    scope_condition: Literal[
        "INCLUDE_ONLY",
        "EXCLUDE",
    ] = "INCLUDE_ONLY",
    resource_orn: str | None = None,
    resource_indicator: str | None = None,
    auth_server_orn: str | None = None,
):
    print(f"Retrieving potential connection for {wlp_id}")

    potential_connections_response = okta_request(
        path="/workload-principals/api/v1/potential-connections",
        method="GET",
        params={
            "filter": f'connectionType eq "{connection_type}"',
        },
    )

    potential_connections = cast(
        utils.types.ResourceConnectionsResponse,
        potential_connections_response.json(),
    )

    potential_connection = next(
        (connection for connection in potential_connections["data"] if connection["connectionType"] == connection_type and ((connection.get("authorizationServer") or {}).get("orn") == auth_server_orn if auth_server_orn is not None else resource_orn is not None and (connection.get("resource") or {}).get("orn") == resource_orn)),
        None,
    )

    if not potential_connection:
        print(f"ℹ️ No available potential connection found for {wlp_id}, skipping creation!")
        return
    else:
        print(f"ℹ️ Potential connection found for {wlp_id}, creating actual connection.")

    print(f"⌛️ Creating {connection_type} connection for {wlp_id}")

    okta_request(
        path=f"/workload-principals/api/v1/ai-agents/{wlp_id}/connections",
        method="POST",
        json={
            "connectionType": connection_type,
            "scopeCondition": scope_condition,
            "scopes": scopes or [],
            "resource": (
                {
                    "orn": resource_orn,
                }
                if (connection_type == "IDENTITY_ASSERTION_A2A_SERVER" and resource_orn is not None)
                else None
            ),
            "authorizationServer": ({"orn": auth_server_orn} if auth_server_orn is not None else None),
            "resourceIndicator": resource_indicator if resource_indicator is not None else potential_connection.get("resourceIndicator"),
        },
    )

    print(f"✅ Connection created successfully!")


# ========== END OF HELPER FUNCTIONS ==========
assert isinstance(
    client_orn := STATE.get(
        "client_app",
        {},
    )
    .get(
        "app",
        {},
    )
    .get("orn"),
    str,
), "❌ Missing client ORN! Make sure you have completed STEP 3."
assert isinstance(
    auth_server_orn := STATE.get(
        "client_app",
        {},
    )
    .get(
        "resource_server",
        {},
    )
    .get("orn"),
    str,
), "❌ Missing client auth server ORN! Make sure you have completed STEP 3."
assert isinstance(
    wlp_id := STATE.get("output").get("PRINCIPAL_A_ID"),
    str,
), "❌ Missing PRINCIPAL_A_ID! Make sure you have completed STEP 5."
assert isinstance(
    resource_orn := STATE.get(
        "agent_j",
        {},
    ).get("orn"),
    str,
), "❌ Missing resource server ORN! Make sure you have completed STEP 5."


create_a2a_resource(
    wlp_id=wlp_id,
    auth_server_orn=auth_server_orn,
)
create_delegation_link(
    client_orn=client_orn,
    resource_orn=resource_orn,
)

print(f"✅ Agent caller successfully added to the workload principal!")

## Step 6: Register Agent K

TODO


### 1. Create the agent
https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>
# @markdown You can change these values but we do not recommend it as it might make the setup inconsistent with the rest of the notebook.

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

import time

PRINCIPAL_B_NAME = "Agent K"  # @param { type: "string", placeholder: "Enter a name for Principal B." }
# @markdown <sup><code>profile.name</code></sup>
PRINCIPAL_B_DESCRIPTION = "Special Agent K"  # @param { type: "string", placeholder: "Enter a description for Principal B." }
# @markdown <sup><code>profile.description</code></sup>
PRINCIPAL_B_RESOURCE_URI = "https://agent_k"  # @param { type: "string", placeholder: "Enter the resource URI for Principal B." }

agent_k = STATE.get("agent_k")
if agent_k is None:
    agent_k = cast(
        utils.types.AIAgentState,
        {},
    )
    STATE["agent_k"] = agent_k

try:
    agent_k = cast(
        utils.types.AIAgentState,
        register_agent(
            agent_k,
            name=PRINCIPAL_B_NAME,
            description=PRINCIPAL_B_DESCRIPTION,
            resource_uri=PRINCIPAL_B_RESOURCE_URI,
        ),
    )
    STATE["agent_k"] = agent_k
finally:
    if agent_k.get("id") != STATE["output"].get("PRINCIPAL_B_ID"):
        STATE["output"]["PRINCIPAL_B_SECRET"] = ""
    STATE["output"]["PRINCIPAL_B_ID"] = agent_k.get(
        "id",
        "",
    )
    STATE["output"]["PRINCIPAL_B_RESOURCE_URI"] = agent_k.get(
        "resource_uri",
        "",
    )

print("   ‼️ Copy the following values to your A2A Colab Notebook!")
print(f"    → PRINCIPAL_B_ID: {agent_k.get('id')}")
print(f"    → PRINCIPAL_B_RESOURCE_URI: {agent_k.get('resource_uri')}")

### 2. Configure client registration

TODO

https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm#:~:text=Add%20client%20registration%20details

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>
# @markdown <br><br>
# @markdown You can change these values but we do not recommend it as it might make the setup inconsistent with the rest of the notebook.
# @markdown <br><br>
# @markdown <em>Only <code>client_secret_basic</code> is supported at this time.</em>
# @markdown <br><br>
# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

PRINCIPAL_B_AUTH_METHOD = "client_secret_basic"  # @param ["client_secret_basic"]
# @markdown <sup><code>client.token_endpoint_auth_method</code></sup>

assert isinstance(
    principal_b_id := STATE["output"].get("PRINCIPAL_B_ID"),
    str,
), "PRINCIPAL_B_ID is missing!"

principal_b_secret = create_credentials(principal_b_id)
STATE["output"]["PRINCIPAL_B_SECRET"] = principal_b_secret

print(f"   ‼️ Copy the following values to your A2A Colab Notebook!")
print(f"     → PRINCIPAL_B_SECRET:")
print(f"         {principal_b_secret}")

### 3. Activate Agent K

TODO

https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm#:~:text=Activate%20an%20AI%20agent



In [ ]:
assert isinstance(
    principal_b_id := STATE["output"].get("PRINCIPAL_B_ID"),
    str,
), "PRINCIPAL_B_ID is missing!"

activate_agent(principal_b_id)

## Step 7: Create a Agent K's Authorization Server

TODO


### 1. Create a custom authorization Server

TODO

https://help.okta.com/oie/en-us/content/topics/security/api-config-auth-server.

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

DOMAIN_B_AUTH_SERVER_NAME = "Agent K"  # @param { type: "string", placeholder: "Enter a name for your Principal/Domain B\'s authorization server." }
# @markdown <sup><code>name</code></sup>

domain_b_resource_server = create_auth_server(
    name=DOMAIN_B_AUTH_SERVER_NAME,
    description=f"{DOMAIN_B_AUTH_SERVER_NAME} Authorization Server",
    audiences=[
        STATE.get(
            "output",
            {},
        ).get(
            "PRINCIPAL_B_RESOURCE_URI",
            "https://agent_b",
        )
    ],
)

# Generate ORN. We will need it later.
if not domain_b_resource_server.get("orn"):
    domain_b_resource_server.update(
        {
            "orn": get_orn(
                service="idp",
                object_type="authorization_servers",
                object_id=domain_b_resource_server.get("id"),
            )
        }
    )

STATE["agent_k"]["resource_server"] = domain_b_resource_server
STATE["output"]["DOMAIN_B_ISSUER"] = domain_b_resource_server.get("issuer")


print("   ‼️ Copy the following values to your A2A Colab Notebook!")
print(f"    → DOMAIN_B_ISSUER: {domain_b_resource_server.get('issuer')}")
print(f"       (or use the auth server ID): {domain_b_resource_server.get('id')}")

### 2. Create a custom scope

TODO

https://help.okta.com/oie/en-us/content/topics/security/api-config-scopes.htm

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

# @markdown <sub><em>Separate multiple scope names with commas.</em></sub>
DOMAIN_B_AUTH_SERVER_SCOPES = "mcp:read"  # @param { type: "string", placeholder: "Enter scope names for your Principal/Domain B\'s required scopes." }
# @markdown <sup><code>name</code> <code>displayName</code></sup>
# @markdown <br><sub><em>Separate multiple scope descriptions with commas in the same order as the name(s)</em></sub>
DOMAIN_B_AUTH_SERVER_SCOPE_DESCRIPTIONS = "Read MCP data"  # @param { type: "string", placeholder: "Enter a description for your Principal/Domain B\'s required scopes." }
# @markdown <sup><code>description</code></sup>


if (domain_b_resource_server := STATE["agent_k"].get("resource_server")) is None:
    raise ValueError("❌ Resource server must be created before setting up its scopes. " "Rerun the authorization server setup cell.")

domain_b_scopes = create_scopes(
    auth_server=domain_b_resource_server,
    scopes=DOMAIN_B_AUTH_SERVER_SCOPES,
    descriptions=DOMAIN_B_AUTH_SERVER_SCOPE_DESCRIPTIONS,
    displayNames=DOMAIN_B_AUTH_SERVER_SCOPES,
)
# This is a bit counter-intuitive but go with it. We are storing Domain B's scopes under PRINCIPAL_A_SCOPES because PRINCIPAL A will be using them to call Principal B.
STATE["output"]["PRINCIPAL_A_SCOPES"] = domain_b_scopes

print(f"   ‼️ Copy the following values to your A2A Colab Notebook!")
print(f"     → PRINCIPAL_A_SCOPES: {STATE['output'].get('PRINCIPAL_A_SCOPES')}")

### 3. Create an access policy & rule

TODO

https://help.okta.com/oie/en-us/content/topics/security/api-config-access-policies.htm

In [ ]:
# @title { display-mode: "form" }
# @markdown <em>Below each input field you will see the corresponding key the value will be mapped to in the API call to Okta.</em>

# @markdown <sup><em>Don't forget to click ![run](https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/images/icons/colab_play_md.png) to run the cell after entering the values!</em></sup>

DOMAIN_B_AUTH_SERVER_POLICY_NAME = "Domain B Policy"  # @param { type: "string", placeholder: "Enter a name for your Principal/Domain B\'s access policy." }
# @markdown <sup><code>name</code> <code>displayName</code></sup>
DOMAIN_B_AUTH_SERVER_POLICY_DESCRIPTION = "Policy for Client App authorization server"  # @param { type: "string", placeholder: "Enter a description for your Principal/Domain B\'s access policy rule." }
DOMAIN_B_AUTH_SERVER_RULE_NAME = "Invoke Agent"  # @param { type: "string", placeholder: "Enter a name for your Client/Principal/Domain B\'s access policy rule." }
# @markdown <sup><code>description</code></sup>

if (domain_b_resource_server := STATE["agent_k"].get("resource_server")) is None:
    raise ValueError("Resource server not found. Create it first.")

domain_b_resource_server_policy_id = create_policy(
    auth_server=domain_b_resource_server,
    name=DOMAIN_B_AUTH_SERVER_POLICY_NAME,
    description=DOMAIN_B_AUTH_SERVER_POLICY_DESCRIPTION,
    rule_name=DOMAIN_B_AUTH_SERVER_RULE_NAME,
    scopes=[
        scope.strip()
        for scope in STATE["output"]
        .get(
            "PRINCIPAL_A_SCOPES",
            "[*]",
        )
        .strip()
        .strip("[]")
        .split(",")
        if scope.strip()
    ],
)

domain_b_resource_server["resource_access_policy_id"] = domain_b_resource_server_policy_id

STATE["agent_k"]["resource_server"] = domain_b_resource_server

### 4. Configure Agent Caller

TODO

https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-add-manually.htm#:~:text=Configure%20machine%20access

In [ ]:
# @title { display-mode: "form" }
# @markdown Helper functions for creating a delegation link. Click 'Show Code' to see the implementation.

assert isinstance(
    auth_server_orn := STATE.get(
        "agent_k",
        {},
    )
    .get(
        "resource_server",
        {},
    )
    .get("orn"),
    str,
), "❌ Missing auth server ORN! Make sure you completed the prior step(s)."
assert isinstance(
    wlp_id := STATE.get(
        "agent_k",
        {},
    ).get("id"),
    str,
), "❌ Missing Principal B ID! Make sure you have completed STEP 6.1."


create_a2a_resource(
    wlp_id=wlp_id,
    auth_server_orn=auth_server_orn,
)

assert isinstance(
    agent_j_orn := STATE.get(
        "agent_j",
        {},
    ).get("orn"),
    str,
), "❌ Missing Principal A ORN! Make sure you have completed STEP 5.1."
assert isinstance(
    agent_k_orn := STATE.get(
        "agent_k",
        {},
    ).get("orn"),
    str,
), "❌ Missing Principal B ORN! Make sure you have completed STEP 6.1."

create_delegation_link(
    client_orn=agent_j_orn,
    resource_orn=agent_k_orn,
    auth_server_orn=auth_server_orn,
)

print(f"✅ Agent caller successfully added to the workload principal!")

## 8. Configure Agent J Connection

TODO

Because Agent K did not exist when we setup Agent J we were unable to add Agent K as a connection. Now that we have created Agent K and configured it to be Agent J's delegate, we need to also add Agent K as a connection [to Agent J]. 

*Adding Agent J as a caller via the Okta Admin Dashboard would normally automatically create a connection for Agent J making this step unnecessary.*

https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-secure.htm

In [ ]:
# @title { display-mode: "form" }
# @markdown Helper functions creating agent connections. Click 'Show Code' to see the implementation.


assert isinstance(
    auth_server_orn := STATE.get(
        "agent_k",
        {},
    )
    .get(
        "resource_server",
        {},
    )
    .get("orn"),
    str,
), "❌ Missing auth server ORN! Make sure you have completed STEP 7."
assert isinstance(
    wlp_id := STATE.get(
        "agent_j",
        {},
    ).get("id"),
    str,
), "❌ Missing Principal A ID! Make sure you have completed STEP 5.1."
assert isinstance(
    principal_a_scopes := STATE.get(
        "output",
        {},
    ).get("PRINCIPAL_A_SCOPES"),
    str,
), "❌ Missing Principal A scopes! Make sure you have completed the previous steps."
assert isinstance(
    principal_b_id := STATE.get(
        "output",
        {},
    ).get("PRINCIPAL_B_ID"),
    str,
), "❌ Missing Principal B ID! Make sure you have completed the previous steps."

create_connection(
    wlp_id=wlp_id,
    connection_type="IDENTITY_ASSERTION_A2A_SERVER",
    scopes=[s.strip() for s in principal_a_scopes.split(",")],
    resource_orn=get_orn(
        service="directory",
        object_type="resource-servers",
        app_name="a2a",
        object_id=principal_b_id,
    ),
    auth_server_orn=auth_server_orn,
)

## 9. Add 'HR Benefits MCP' as a resource connection 

https://help.okta.com/oie/en-us/content/topics/ai-agents/ai-agent-secure.htm

In [ ]:
assert isinstance(
    auth_server_orn := STATE.get(
        "resource",
        {},
    ).get("orn"),
    str,
), "❌ Missing auth server ORN! Make sure you have completed STEP 7."
print("Auth server ORN:", auth_server_orn)
assert isinstance(
    wlp_id := STATE.get(
        "agent_k",
        {},
    ).get("id"),
    str,
), "❌ Missing Principal B ID! Make sure you have completed Agent K Registration."
assert isinstance(
    principal_b_scopes := STATE.get(
        "output",
        {},
    ).get("PRINCIPAL_B_SCOPES"),
    str,
), "❌ Missing Principal B scopes! Make sure you have completed the previous steps."

create_connection(
    wlp_id=wlp_id,
    connection_type="IDENTITY_ASSERTION_CUSTOM_AS",
    scopes=[s.strip() for s in principal_b_scopes.split(",")],
    # resource_orn=get_orn(
    #     service="directory",
    #     object_type="resource-servers",
    #     app_name="a2a",
    #     object_id=principal_b_id,
    # ),
    auth_server_orn=auth_server_orn,
)

## 10. Setup complete!

**Return to your A2A Colab and initialize the configuration to validate your setup.**

In [ ]:
setup_output = STATE.get("output", {})


def as_scope_list(value):
    if isinstance(value, list):
        return value
    return (value or "").replace(",", " ").split()


cell9_config = {
    "OKTA_DOMAIN": OKTA_DOMAIN,
    "DOMAIN_A_ISSUER": setup_output.get("DOMAIN_A_ISSUER", ""),
    "REDIRECT_URI": setup_output.get("REDIRECT_URI", ""),
    "CLIENT_ID": setup_output.get("CLIENT_ID", ""),
    "CLIENT_SCOPES": as_scope_list(setup_output.get("CLIENT_SCOPES")),
    "CLIENT_SECRET": setup_output.get("CLIENT_SECRET", ""),
    "CLIENT_PRIVATE_JWK": setup_output.get("CLIENT_PRIVATE_JWK", {}),
    "PRINCIPAL_A_ID": setup_output.get("PRINCIPAL_A_ID", ""),
    "PRINCIPAL_A_RESOURCE_URI": setup_output.get("PRINCIPAL_A_RESOURCE_URI", ""),
    "PRINCIPAL_A_SCOPES": as_scope_list(setup_output.get("PRINCIPAL_A_SCOPES")),
    "PRINCIPAL_A_SECRET": setup_output.get("PRINCIPAL_A_SECRET", ""),
    "PRINCIPAL_A_PRIVATE_JWK": setup_output.get("PRINCIPAL_A_PRIVATE_JWK", {}),
    "DOMAIN_B_ISSUER": setup_output.get("DOMAIN_B_ISSUER", ""),
    "PRINCIPAL_B_RESOURCE_URI": setup_output.get("PRINCIPAL_B_RESOURCE_URI", ""),
    "PRINCIPAL_B_ID": setup_output.get("PRINCIPAL_B_ID", ""),
    "PRINCIPAL_B_SCOPES": as_scope_list(setup_output.get("PRINCIPAL_B_SCOPES")),
    "PRINCIPAL_B_SECRET": setup_output.get("PRINCIPAL_B_SECRET", ""),
    "PRINCIPAL_B_PRIVATE_JWK": setup_output.get("PRINCIPAL_B_PRIVATE_JWK", {}),
    "RESOURCE_ISSUER": setup_output.get("RESOURCE_ISSUER", ""),
    "RESOURCE_SERVER_AUDIENCE": setup_output.get("RESOURCE_SERVER_AUDIENCE", ""),
    "SDK_DEBUG_ENABLED": False,
}

print("Copy these assignments into Cell 9 of cross_app_a2a.ipynb:\n")
for name, value in cell9_config.items():
    print(f"{name} = {value!r}")

### UNDO EVERYTHING

TODO

In [ ]:
1.0